# HDFC Bank: Fraud Risk Analysis - Part 6

**Objective:** Senior ML Engineer Upgrade & Hyperparameter Tuning (Optuna)

We have upgraded our pipeline to use **RobustScaler**, **Log1p transforms**, **Missing Indicators**, and the incredibly powerful **TargetEncoder**. Now, we will use Optuna to mathematically search for the perfect XGBoost hyperparameters to maximize PR-AUC.

In [1]:
import pandas as pd
import numpy as np
import mlflow
import optuna
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

from fraudguard.data.ingestion import load_bank_data, split_temporal
from fraudguard.features.engineering import build_feature_pipeline
from fraudguard.models.training import train_xgboost
from fraudguard.models.evaluation import evaluate_model
from sklearn.pipeline import Pipeline

# MLflow Setup
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("FraudGuard_Optuna_Tuning")

<Experiment: artifact_location='mlflow-artifacts:/668470780902146092', creation_time=1788280630789, effective_trace_archival_retention=None, experiment_id='668470780902146092', last_update_time=1788280630789, lifecycle_stage='active', name='FraudGuard_Optuna_Tuning', tags={}, trace_location=None, workspace='default'>

## 1. Data Preparation with Target Encoding

In [2]:
# 1. Load Data
data_dir = Path.cwd().parent / "data" / "raw"
df = load_bank_data(data_dir)

# 2. Split Temporally
df_train, df_test = split_temporal(df, test_ratio=0.2)
X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud'].values
X_test = df_test.drop(columns=['isFraud'])
y_test = df_test['isFraud'].values

gateway_numeric_features = ['TransactionAmt', 'dist1', 'dist2']
gateway_categorical_features = [
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo'
]

# Ensure features exist
gateway_numeric_features = [f for f in gateway_numeric_features if f in X_train.columns]
gateway_categorical_features = [f for f in gateway_categorical_features if f in X_train.columns]

# 3. Build and Apply Pipeline (CRITICAL: TargetEncoder requires y_train!)
pipeline = build_feature_pipeline(gateway_numeric_features, gateway_categorical_features)
print("Fitting pipeline (RobustScaler + TargetEncoder)...")
X_train_processed = pipeline.fit_transform(X_train, y_train)
X_test_processed = pipeline.transform(X_test)
print(f"Processed Matrix Shape: {X_train_processed.shape}")

Fitting pipeline (RobustScaler + TargetEncoder)...
Processed Matrix Shape: (472432, 47)


## 2. Optuna Objective Function

In [ ]:
def objective(trial):
    # Search Space
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    # Train model
    xgb_model = train_xgboost(
        X_train_processed, y_train, 
        X_val=X_test_processed, y_val=y_test,
        **params
    )
    
    # Evaluate
    xgb_probs = xgb_model.predict_proba(X_test_processed)[:, 1]
    metrics = evaluate_model(y_test, xgb_probs, threshold=0.5)
    
    # We want Optuna to MAXIMIZE the PR-AUC
    return metrics['pr_auc']

## 3. Run Optimization & Log Final Champion

In [4]:
print("Running Optuna Hyperparameter Search (10 Trials for demonstration)...")
study = optuna.create_study(direction="maximize", study_name="XGBoost_Gateway_Tuning")
study.optimize(objective, n_trials=10)

print("\nBest PR-AUC:", study.best_value)
print("Best Parameters:", study.best_params)

# Train the absolute final Champion model with the best parameters
with mlflow.start_run(run_name="XGBoost_Champion_Tuned"):
    mlflow.log_params(study.best_params)
    
    final_model = train_xgboost(
        X_train_processed, y_train, 
        X_val=X_test_processed, y_val=y_test,
        **study.best_params
    )
    
    final_probs = final_model.predict_proba(X_test_processed)[:, 1]
    final_metrics = evaluate_model(y_test, final_probs, threshold=0.5)
    mlflow.log_metrics(final_metrics)
    
    full_deployable_model = Pipeline(steps=[
        ('preprocessor', pipeline),
        ('classifier', final_model)
    ])
    
    mlflow.sklearn.log_model(full_deployable_model, "gateway_champion")
    print("\nFinal Tuned Champion Model Logged to MLflow!")

Running Optuna Hyperparameter Search (10 Trials for demonstration)...


[I 2026-09-01 22:14:54,652] A new study created in memory with name: XGBoost_Gateway_Tuning


[0]	validation_0-aucpr:0.12054
[50]	validation_0-aucpr:0.19559
[100]	validation_0-aucpr:0.20471
[150]	validation_0-aucpr:0.21423
[200]	validation_0-aucpr:0.21999
[250]	validation_0-aucpr:0.22322
[300]	validation_0-aucpr:0.22565
[350]	validation_0-aucpr:0.22752
[400]	validation_0-aucpr:0.22994
[450]	validation_0-aucpr:0.23180
[500]	validation_0-aucpr:0.23464
[550]	validation_0-aucpr:0.23481
[600]	validation_0-aucpr:0.23829
[650]	validation_0-aucpr:0.23809
[700]	validation_0-aucpr:0.24104
[749]	validation_0-aucpr:0.23990


[I 2026-09-01 22:16:01,462] Trial 0 finished with value: 0.24157674881466487 and parameters: {'max_depth': 5, 'learning_rate': 0.07795857747850055, 'subsample': 0.700209868558688, 'colsample_bytree': 0.9685882901543739, 'min_child_weight': 3}. Best is trial 0 with value: 0.24157674881466487.


[0]	validation_0-aucpr:0.17909
[50]	validation_0-aucpr:0.24793
[100]	validation_0-aucpr:0.25703
[150]	validation_0-aucpr:0.25964
[173]	validation_0-aucpr:0.25781


...

[I 2026-09-01 22:24:06,603] Trial 9 finished with value: 0.2419730730803572 and parameters: {'max_depth': 6, 'learning_rate': 0.08330364066609731, 'subsample': 0.5313009827733182, 'colsample_bytree': 0.8383037220390509, 'min_child_weight': 6}. Best is trial 7 with value: 0.28630585458843866.



Best PR-AUC: 0.28630585458843866
Best Parameters: {'max_depth': 9, 'learning_rate': 0.2307895196484926, 'subsample': 0.9675615702591527, 'colsample_bytree': 0.8045779978175522, 'min_child_weight': 7}
[0]	validation_0-aucpr:0.18264
[50]	validation_0-aucpr:0.25416
[100]	validation_0-aucpr:0.26031
[150]	validation_0-aucpr:0.26512
[200]	validation_0-aucpr:0.26836
[250]	validation_0-aucpr:0.27143
[300]	validation_0-aucpr:0.27478
[350]	validation_0-aucpr:0.27496
[400]	validation_0-aucpr:0.27518
[450]	validation_0-aucpr:0.27805
[500]	validation_0-aucpr:0.27963
[550]	validation_0-aucpr:0.28215
[600]	validation_0-aucpr:0.28360
[650]	validation_0-aucpr:0.28378
[700]	validation_0-aucpr:0.28499
[750]	validation_0-aucpr:0.28500
[773]	validation_0-aucpr:0.28546


2026/09/01 22:25:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost_Champion_Tuned at: http://127.0.0.1:5000/#/experiments/668470780902146092/runs/0617f909ba944dada5d514e904c2328e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/668470780902146092


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: '<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>
'